In [1]:
import os

os.makedirs("templates", exist_ok=True)

In [21]:
html_code = """<!DOCTYPE html>
<html>
<head>
    <title>Customer Panel</title>
    <style>
        body { font-family: Arial; padding: 20px; }
        section { margin-bottom: 30px; border-bottom: 1px solid #ddd; padding-bottom: 20px; }
        input, button { margin: 5px; padding: 8px; }
        pre { background: #f4f4f4; padding: 10px; border-radius: 5px; color: #333; }
        .success-msg { color: green; font-weight: bold; }
    </style>
</head>
<body>

<h1>Customer System</h1>

<section>
    <h2>1) Search Product by Name</h2>
    <input id="search_name" placeholder="Product Name">
    <button onclick="searchProduct()">Search</button>
    <pre id="search_result"></pre>
</section>

<section>
    <h2>2) View All Products</h2>
    <button onclick="getAllProducts()">Show All</button>
    <pre id="all_products"></pre>
</section>

<section>
    <h2>3) Create Order</h2>
    <input id="order_product" placeholder="Product Name">
    <input id="order_company" placeholder="Company">
    <input id="order_quantity" type="number" placeholder="Quantity">
    <input id="order_price" type="number" placeholder="Price">

    <button onclick="createOrder()">Create Order</button>
    <pre id="order_result"></pre>
</section>

<section>
    <h2>4) Search Order by ID</h2>
    <input id="order_id" placeholder="Order ID">
    <button onclick="getOrder()">Search</button>
    <pre id="order_search_result"></pre>
</section>

<section>
    <h2>5) Update Order (Add Product)</h2>
    <input id="update_order_id" placeholder="Order ID">
    <input id="new_product" placeholder="New Product Name">
    <input id="new_company" placeholder="Company Name"> 
    <input id="new_quantity" type="number" placeholder="Quantity">
    <input id="new_price" type="number" placeholder="Price">

    <button onclick="updateOrder()">Update Order</button>
    <pre id="update_result"></pre>
</section>

<script>

// Helper to validate if Product and Company exist in database
async function validateProduct(productName, companyName) {
    const res = await fetch('/products');
    const products = await res.json();
    return products.find(p => 
        p.product_name.toLowerCase() === productName.toLowerCase() && 
        p.company.toLowerCase() === companyName.toLowerCase()
    );
}

// ================= 1. SEARCH PRODUCT =================
function searchProduct() {
    fetch('/products')
    .then(res => res.json())
    .then(data => {
        let name = search_name.value.toLowerCase();
        let result = data.filter(p => p.product_name.toLowerCase().includes(name));
        search_result.innerText = JSON.stringify(result, null, 2);
    });
}

// ================= 2. GET ALL PRODUCTS =================
function getAllProducts() {
    fetch('/products')
    .then(res => res.json())
    .then(data => {
        all_products.innerText = JSON.stringify(data, null, 2);
    });
}

// ================= 3. CREATE ORDER =================
async function createOrder() {
    const prodName = order_product.value;
    const compName = order_company.value;
    const qty = parseInt(order_quantity.value);
    const price = parseFloat(order_price.value);

    // 1. Validate Inputs
    if (qty <= 0 || isNaN(qty)) {
        alert("Quantity must be a positive number.");
        return;
    }
    if (price < 0 || isNaN(price)) {
        alert("Price cannot be negative.");
        return;
    }

    // 2. Validate existence
    const productExists = await validateProduct(prodName, compName);
    if (!productExists) {
        alert("Error: The product or company name does not exist in our records.");
        return;
    }

    // 3. Create Request
    fetch('/order', {
        method: 'POST',
        headers: {'Content-Type': 'application/json'},
        body: JSON.stringify({
            date: new Date().toISOString(),
            products: [{
                product_name: prodName,
                company: compName,
                quantity: qty,
                price: price
            }]
        })
    })
    .then(res => res.json())
    .then(data => {
        // Displaying Order ID (ObjectId)
        order_result.innerHTML = `<span class="success-msg">Order Created! ID: ${data._id}</span>\n` + JSON.stringify(data, null, 2);
    });
}

// ================= 4. SEARCH ORDER =================
function getOrder() {
    let id = order_id.value;
    fetch('/order/' + id)
    .then(res => res.json())
    .then(data => {
        order_search_result.innerText = JSON.stringify(data, null, 2);
    });
}

// ================= 5. UPDATE ORDER =================
async function updateOrder() {
    let id = update_order_id.value;
    const prodName = new_product.value;
    const compName = new_company.value;
    const qty = parseInt(new_quantity.value);
    const price = parseFloat(new_price.value);

    // 1. Validate Inputs
    if (qty <= 0 || isNaN(qty)) {
        alert("Quantity must be a positive number.");
        return;
    }
    if (price < 0 || isNaN(price)) {
        alert("Price cannot be negative.");
        return;
    }

    // 2. Validate existence
    const productExists = await validateProduct(prodName, compName);
    if (!productExists) {
        alert("Error: The product or company name does not exist.");
        return;
    }

    // 3. Get existing order and update
    fetch('/order/' + id)
    .then(res => res.json())
    .then(order => {
        if (!order || !order.products) {
            alert("Order not found");
            return;
        }

        order.products.push({
            product_name: prodName,
            company: compName,
            quantity: qty,
            price: price
        });

        fetch('/order/' + id, {
            method: 'PUT',
            headers: {'Content-Type': 'application/json'},
            body: JSON.stringify(order)
        })
        .then(res => res.json())
        .then(data => {
            update_result.innerHTML = `<span class="success-msg">Order ${data._id} Updated!</span>\n` + JSON.stringify(data, null, 2);
        });
    });
}

</script>
</body>
</html>
"""

with open("templates/index.html", "w") as f:
    f.write(html_code)

In [19]:
#if i had to make changes in this front-end design
import os

os.remove("templates/index.html")